# 04 — Parametric & Non-Parametric Statistical Analysis
**Project:** Axlero Solutions — Credit Risk Scoring and Loan Default Analysis  
**Module:** Statistical Hypothesis Testing (Module 3 — Charukesh)

## 1. Methodology
Because financial variables (`person_income`, `loan_amnt`, `person_emp_length`) exhibit strong positive skewness, we pair parametric tests (**Welch's two-sample t-test**, **Point-Biserial correlation**) with robust non-parametric counterparts:
- **Mann-Whitney U test** and **Rank-Biserial Correlation ($r_{rb} = \frac{2U}{n_1 n_0} - 1$)**
- **Two-Sample Kolmogorov-Smirnov (KS) Statistic ($D_{KS} = \sup_x |F_1(x) - F_0(x)|$)**
- **Pearson's Chi-Square ($\chi^2$) Test of Independence** and **Cramér's V ($V = \sqrt{\frac{\chi^2}{n \cdot \min(r-1, c-1)}}$)** with **95% Wilson Score Confidence Intervals** and **Odds Ratios**.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_validated_data
from src.features.statistical_tests import analyze_numerical_features, analyze_categorical_features
from src.utils import load_config

cfg = load_config()
df = load_validated_data()
num_cols = cfg["data_schema"]["numeric_columns"]
cat_cols = cfg["data_schema"]["categorical_columns"]

num_stats = analyze_numerical_features(df, num_cols)
print("Numerical Variables — Hypothesis Test & Effect Size Summary (Sorted by KS Statistic):")
display(num_stats)

Numerical Variables — Hypothesis Test & Effect Size Summary (Sorted by KS Statistic):


,feature,overall_mean,overall_sd,overall_median,overall_iqr,skewness,kurtosis,repaid_mean,repaid_median,default_mean,...,welch_t_stat,welch_p_value,mann_whitney_u,mann_whitney_p_value,rank_biserial_r,ks_statistic,ks_p_value,spearman_r,point_biserial_r,significant_at_0_01
0,loan_percent_income,0.1702,0.1068,0.15,0.14,1.0638,1.2173,0.1488,0.13,0.2469,...,59.0320,0.000000e+00,129439860.0,0.000000e+00,0.4419,0.3519,0.000000e+00,0.3165,0.3797,True
1,loan_int_rate,11.0202,3.2134,10.99,5.60,0.2100,-0.6771,10.4521,10.65,13.0497,...,60.2776,0.000000e+00,129703191.5,0.000000e+00,0.4448,0.3340,0.000000e+00,0.3185,0.3341,True
2,person_income,66091.6408,62015.5803,55000.00,40676.00,32.9572,2700.6994,70849.1228,60000.00,49094.4980,...,-36.1581,6.044839e-278,55642822.5,0.000000e+00,-0.3802,0.2802,0.000000e+00,-0.2722,-0.1450,True
3,loan_amnt,9593.8456,6322.7302,8000.00,7250.00,1.1919,1.4208,9240.1558,8000.00,10857.4799,...,17.4069,6.920493e-67,100354119.0,3.189187e-52,0.1179,0.1172,9.788345e-67,0.0845,0.1057,True
4,person_emp_length,4.7615,3.9836,4.00,5.00,1.2810,2.5880,4.9427,4.00,4.1141,...,-16.0554,2.092752e-57,77766892.5,3.627512e-67,-0.1337,0.1047,2.827709e-53,-0.0962,-0.0860,True
5,person_age,27.7301,6.2100,26.00,7.00,1.9425,5.8676,27.8017,26.00,27.4744,...,-3.8976,9.771256e-05,85451813.0,4.885222e-10,-0.0481,0.0355,1.678810e-06,-0.0346,-0.0218,True
6,cb_person_cred_hist_length,5.8113,4.0590,4.00,5.00,1.6593,3.7051,5.8467,4.00,5.6847,...,-2.9464,3.221191e-03,86673103.0,7.036255e-06,-0.0345,0.0307,5.875820e-05,-0.0250,-0.0165,True


## 2. Categorical Variables — Chi-Square, Cramér's V, Wilson 95% CIs & Odds Ratios

In [2]:
cat_stats, level_tables = analyze_categorical_features(df, cat_cols)
print("Categorical Variables — Omnibus Chi-Square & Cramér's V Summary:")
display(cat_stats)

for col, tbl in level_tables.items():
    print(f"\nLevel-Wise Default Risk & Odds Ratios: {col}")
    display(tbl)

Categorical Variables — Omnibus Chi-Square & Cramér's V Summary:


,feature,num_categories,chi2_statistic,degrees_of_freedom,p_value,cramers_v,effect_interpretation,significant_at_0_01
0,loan_grade,7,5588.3264,6,0.000000e+00,0.4152,Strong (>0.30),True
1,person_home_ownership,4,1894.3527,3,0.000000e+00,0.2417,Moderate (0.15-0.30),True
2,cb_person_default_on_file,2,1041.9738,1,1.351644e-228,0.1793,Moderate (0.15-0.30),True
3,loan_intent,6,520.1814,5,3.512382e-110,0.1267,Weak (<0.15),True



Level-Wise Default Risk & Odds Ratios: person_home_ownership


,category,non_default,default,total,share_pct,default_rate_pct,ci_95_low_pct,ci_95_high_pct,odds_ratio_vs_lowest
0,OWN,2371,192,2563,7.91,7.49,6.53,8.58,1.00
1,MORTGAGE,11682,1687,13369,41.24,12.62,12.07,13.19,1.78
2,OTHER,73,33,106,0.33,31.13,23.11,40.48,5.62
3,RENT,11201,5177,16378,50.52,31.61,30.90,32.33,5.69



Level-Wise Default Risk & Odds Ratios: loan_intent


,category,non_default,default,total,share_pct,default_rate_pct,ci_95_low_pct,ci_95_high_pct,odds_ratio_vs_lowest
0,VENTURE,4838,844,5682,17.53,14.85,13.95,15.80,1.00
1,EDUCATION,5305,1106,6411,19.78,17.25,16.35,18.20,1.19
2,PERSONAL,4404,1094,5498,16.96,19.90,18.86,20.97,1.42
3,HOMEIMPROVEMENT,2654,940,3594,11.09,26.15,24.74,27.62,2.03
4,MEDICAL,4425,1617,6042,18.64,26.76,25.66,27.89,2.09
5,DEBTCONSOLIDATION,3701,1488,5189,16.01,28.68,27.46,29.92,2.30



Level-Wise Default Risk & Odds Ratios: loan_grade


,category,non_default,default,total,share_pct,default_rate_pct,ci_95_low_pct,ci_95_high_pct,odds_ratio_vs_lowest
0,A,9637,1066,10703,33.02,9.96,9.41,10.54,1.00
1,B,8692,1695,10387,32.04,16.32,15.62,17.04,1.76
2,C,5102,1336,6438,19.86,20.75,19.78,21.76,2.37
3,D,1482,2138,3620,11.17,59.06,57.45,60.65,13.04
4,E,342,621,963,2.97,64.49,61.41,67.45,16.40
5,F,71,170,241,0.74,70.54,64.50,75.94,21.55
6,G,1,63,64,0.20,98.44,91.67,99.72,382.55



Level-Wise Default Risk & Odds Ratios: cb_person_default_on_file


,category,non_default,default,total,share_pct,default_rate_pct,ci_95_low_pct,ci_95_high_pct,odds_ratio_vs_lowest
0,N,21767,4919,26686,82.32,18.43,17.97,18.90,1.0
1,Y,3560,2170,5730,17.68,37.87,36.62,39.13,2.7
